In [13]:
import pandas as pd
import requests
from datetime import datetime

# Connect with API
base_url = "https://data.cityofnewyork.us/api/v3/views/k46n-sa2m/query.json?pageNumber=1&pageSize=2000&app_token=wlEqLX8cddLmKefHSpYZ8qgbK"
response = requests.get(base_url)

# Set up columns via lists
date = []
total_adults = []
total_children = []
total_individuals = []
single_men = []
single_women = []
total_single_adults = []
families_with_children = []
adults_in_families = []
children = []
total_individuals_families = []
adult_families = []
individuals_adult_familes = []

# If status code is valid, append the values into the lists above and convert them into columns
if response.status_code == 200:
    dhs_data = response.json()
    for row in dhs_data:

        # Specific edits for date to get only year, month, day and convert from string to date
        date.append(pd.to_datetime(row["date_of_census"][0:10]))

        total_adults.append(int(row["total_adults_in_shelter"]))  
        total_children.append(int(row["total_children_in_shelter"]))
        total_individuals.append(int(row["total_individuals_in_shelter"]))
        single_men.append(int(row["single_adult_men_in_shelter"]))
        single_women.append(int(row["single_adult_women_in_shelter"]))
        families_with_children.append(int(row["families_with_children_in_shelter"]))
else:
    print("ERROR")

# Create dataframe for all values and set up columns
df = pd.DataFrame({"Date": date, "Adults": total_adults, "Single Men": single_men, "Single Women": single_women, "Children": total_children, "Families": families_with_children, "Individuals": total_individuals})
df_adults = df[["Adults", "Date"]]
df

,Date,Adults,Single Men,Single Women,Children,Families,Individuals
0,2021-03-01,35195,13936,4543,16746,9593,51941
1,2021-03-02,35202,13962,4526,16746,9590,51948
2,2021-03-03,35108,13931,4503,16720,9579,51828
3,2021-03-04,35176,13989,4526,16720,9578,51896
4,2021-03-05,35103,13939,4485,16766,9598,51869
...,...,...,...,...,...,...,...
1927,2026-06-10,54150,17982,7296,28044,16260,82194
1928,2026-06-11,54360,18087,7308,28173,16330,82533
1929,2026-06-12,54255,18014,7249,28180,16346,82435
1930,2026-06-13,54361,18026,7278,28223,16373,82584


In [ ]:
# Objective: Create dataframe that shows maximum adult value per month/year, and also show the exact date of each month/year

##########
# Group the original dataframe by year and month by maximum value per unique month/year in each column
grouped_df = df.groupby([df['Date'].dt.year, df['Date'].dt.month]).max()

# Dataframe for just maxmimum adult values per month/year from grouped_df. This will contain the maximum values per month/year but will not show the exact date
grouped_adults = grouped_df[["Date", "Adults"]]
#########

#########
# To isolate and find exact date...
# Create specific list for max adult values from grouped_adults
max_adult_values = list(grouped_adults["Adults"])

# Create a new dataframe from the original dataframe of all adult values with the "Adults" value column as the index
adult_series = df_adults.set_index("Adults").squeeze("columns")

# Create a dataframe of only the max values from adult_series by using loc with max_adult_values. This will also include more dates per month since there are common numbers
adult_loc_values = pd.DataFrame(adult_series.loc[max_adult_values]).sort_values(by='Date')

# Isolate duplicate adult values by calling the maximum value per month/year combination to reveal actual date of max adult value per month
adult_final_dates = adult_loc_values.groupby([adult_loc_values['Date'].dt.year, adult_loc_values['Date'].dt.month]).max()
###########

# Combine the dates from adult_final_dates with values from values of grouped_adults. This is now the final result
pd.DataFrame({"Date": list(adult_final_dates["Date"]), "Adults": list(grouped_adults["Adults"])})

,Date,Adults
0,2021-03-07,35232
1,2021-04-01,34471
2,2021-05-05,33638
3,2021-06-01,32826
4,2021-07-16,31326
...,...,...
59,2026-02-08,56265
60,2026-03-02,55895
61,2026-04-13,55171
62,2026-05-23,54913


In [20]:
# Do same as above but written as a function so all columns can be entered

def find_maxval_date(column):
    grouped_dataframe = df.groupby([df['Date'].dt.year, df['Date'].dt.month]).max()
    grouped_column = grouped_dataframe[["Date", f"{column}"]]
    max_column_values = list(grouped_column[f"{column}"])
    column_series = df[[f"{column}", "Date"]].set_index(f"{column}").squeeze("columns")
    column_loc_values = pd.DataFrame(column_series.loc[max_column_values]).sort_values(by='Date')
    column_final_dates = column_loc_values.groupby([column_loc_values['Date'].dt.year, column_loc_values['Date'].dt.month]).max()
    return pd.DataFrame({"Date": list(column_final_dates["Date"]), f"{column}": list(grouped_column[f"{column}"])})

find_maxval_date("Families")


,Date,Families
0,2021-03-07,9624
1,2021-04-04,9299
2,2021-05-01,9042
3,2021-06-06,8847
4,2021-07-04,8333
...,...,...
59,2026-02-01,17548
60,2026-03-30,17314
61,2026-04-05,16850
62,2026-05-04,16635


In [21]:
# THROWAWAY CODE

# Testing if dates work
# df[df["Date"] >= "2026-1-1"]
# df[df["Date"] > "2025"]
# print(type(df["Date"][1]))

# test = df[df["Date"].between("2021-12-1", "2022-1")]
# test

# Display or don't display all rows
# pd.set_option('display.max_rows', None)
# pd.reset_option("display.max_rows")

# Grouped by year and month, then created dataframe for maximum value per unique month/year in each column
# grouped_df = df.groupby([df['Date'].dt.year, df['Date'].dt.month]).max()

#pd.set_option('display.max_rows', None)

# Maximum values for adults
# grouped_adults = grouped_df[["Date", "Adults"]]
# max_adult_values = list(grouped_adults["Adults"])

# adult_series = df_adults.set_index("Adults").squeeze("columns")
# pd.DataFrame(adult_series.loc[max_adult_values]).sort_values(by='Date')